# Window Function Basics

## Overview
A **window function** computes a value for each row using a set of rows related to the current row — the "window." Unlike `GROUP BY`, window functions **do not collapse rows**. Every input row remains in the output, with the computed value appended as a new column.

**Anatomy of a window function call:**
```sql
function_name(args)
OVER (
    [PARTITION BY partition_columns]
    [ORDER BY order_columns]
    [frame_clause]
)
```

| Clause | Purpose | Optional? |
|---|---|---|
| `PARTITION BY` | Divides rows into independent groups (like GROUP BY for the window) | Yes — omit to use all rows as one window |
| `ORDER BY` | Defines the logical row sequence within each partition | Required for ranking/offset/frame functions; optional for pure aggregates |
| Frame clause | Defines which rows relative to the current row are included | Optional — defaults vary by function |

**Window functions vs GROUP BY:**

| | GROUP BY aggregate | Window function |
|---|---|---|
| Rows in output | One per group | Same as input — all rows preserved |
| Access to individual row values | No | Yes |
| Can reference other columns freely | Only grouped/aggregated ones | Yes |
| Typical use | Totals, counts per group | Rankings, running totals, comparisons |

**SQLite support:** SQLite has supported window functions since version 3.25.0 (2018). All functions in this folder work in SQLite and PostgreSQL unless noted.

---

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT,
    segment TEXT, province TEXT
);
CREATE TABLE accounts (
    account_id INTEGER PRIMARY KEY, customer_id INTEGER,
    account_type TEXT, province TEXT, opened_date TEXT, status TEXT
);
CREATE TABLE transactions (
    txn_id     INTEGER PRIMARY KEY,
    account_id INTEGER,
    txn_date   TEXT,
    txn_type   TEXT,
    amount     REAL,
    category   TEXT,
    flagged    INTEGER
);
CREATE TABLE trades (
    trade_id    INTEGER PRIMARY KEY,
    account_id  INTEGER,
    trade_date  TEXT,
    ticker      TEXT,
    direction   TEXT,   -- Buy / Sell
    shares      INTEGER,
    price       REAL,
    total_value REAL
);

INSERT INTO customers VALUES
  (1,'Aroha','Ngata','Retail','NB'),(2,'Liam','Chen','SME','BC'),
  (3,'Fatima','Al-Rashid','Wealth','ON'),(4,'James','MacLeod','Retail','NB'),
  (5,'Sofia','Petrov','SME','BC'),(6,'Noah','Williams','Retail','AB'),
  (7,'Mei','Zhang','Wealth','ON'),(8,'Ethan','Tremblay','Retail','QC'),
  (9,'Priya','Nair','SME','BC'),(10,'Marcus','Okafor','Retail','ON');

INSERT INTO accounts VALUES
  (101,1,'Chequing','NB','2019-03-15','Active'),
  (102,1,'Savings','NB','2019-03-15','Active'),
  (103,2,'Chequing','BC','2020-07-01','Active'),
  (104,3,'Investment','ON','2018-05-20','Active'),
  (105,4,'Chequing','NB','2015-09-30','Active'),
  (106,5,'Chequing','BC','2021-06-15','Active'),
  (107,6,'Chequing','AB','2017-11-22','Active'),
  (108,7,'Investment','ON','2016-03-08','Active'),
  (109,8,'Chequing','QC','2023-01-05','Active'),
  (110,9,'Investment','BC','2022-08-19','Active'),
  (111,10,'Chequing','ON','2020-12-01','Active');

INSERT INTO transactions VALUES
  (1001,101,'2023-01-05','Deposit',   4200.00,'Payroll',0),
  (1002,101,'2023-01-12','Withdrawal',-850.00,'Rent',0),
  (1003,101,'2023-01-20','Withdrawal',-120.00,'Groceries',0),
  (1004,101,'2023-02-05','Deposit',   4200.00,'Payroll',0),
  (1005,101,'2023-02-18','Withdrawal',-980.00,'Rent',0),
  (1006,101,'2023-03-05','Deposit',   4200.00,'Payroll',0),
  (1007,103,'2023-01-08','Deposit',  15000.00,'Payroll',0),
  (1008,103,'2023-01-25','Withdrawal',-3200.00,'Rent',0),
  (1009,103,'2023-02-08','Deposit',  15000.00,'Payroll',0),
  (1010,103,'2023-02-22','Withdrawal',-2800.00,'Rent',0),
  (1011,103,'2023-03-08','Deposit',  15000.00,'Payroll',0),
  (1012,105,'2023-01-06','Deposit',   3100.00,'Payroll',0),
  (1013,105,'2023-01-19','Withdrawal',-700.00,'Rent',0),
  (1014,105,'2023-02-06','Deposit',   3100.00,'Payroll',0),
  (1015,105,'2023-02-20','Withdrawal',-650.00,'Groceries',0),
  (1016,107,'2023-01-20','Deposit',   3800.00,'Payroll',0),
  (1017,107,'2023-02-10','Fee',         -25.00,'Fee',1),
  (1018,107,'2023-03-15','Withdrawal', -450.00,'Groceries',1),
  (1019,109,'2023-01-10','Deposit',   2800.00,'Payroll',0),
  (1020,109,'2023-01-28','Withdrawal',-650.00,'Groceries',0),
  (1021,111,'2023-01-22','Deposit',   4500.00,'Payroll',0),
  (1022,111,'2023-02-15','Withdrawal',-1100.00,'Utilities',0),
  (1023,111,'2023-03-01','Deposit',   4500.00,'Payroll',0);

INSERT INTO trades VALUES
  (2001,104,'2023-01-10','AAPL','Buy', 10,148.50,1485.00),
  (2002,104,'2023-01-25','MSFT','Buy', 5, 240.10,1200.50),
  (2003,104,'2023-02-14','AAPL','Buy', 5, 153.20, 766.00),
  (2004,104,'2023-02-28','MSFT','Sell',3, 252.80, 758.40),
  (2005,104,'2023-03-15','NVDA','Buy', 2, 278.50, 557.00),
  (2006,108,'2023-01-05','AAPL','Buy',20, 130.50,2610.00),
  (2007,108,'2023-01-18','AMZN','Buy', 8,  95.20, 761.60),
  (2008,108,'2023-02-08','AAPL','Sell',10,151.00,1510.00),
  (2009,108,'2023-02-22','AMZN','Buy', 5,  99.80, 499.00),
  (2010,108,'2023-03-10','NVDA','Buy', 4, 265.30,1061.20),
  (2011,110,'2023-01-12','MSFT','Buy', 8, 238.00,1904.00),
  (2012,110,'2023-02-01','AAPL','Buy',15, 145.00,2175.00),
  (2013,110,'2023-02-20','MSFT','Buy', 3, 248.50, 745.50),
  (2014,110,'2023-03-05','AAPL','Sell',10,155.00,1550.00),
  (2015,110,'2023-03-20','NVDA','Buy', 6, 280.00,1680.00);
""")
conn.commit()

print('Finance schema ready. Table row counts:')
for t in ['customers','accounts','transactions','trades']:
    n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t}: {n} rows')


Finance schema ready. Table row counts:
  customers: 10 rows
  accounts: 11 rows
  transactions: 23 rows
  trades: 15 rows


---
## The core contrast: GROUP BY collapses rows, window functions do not

In [2]:
# Side-by-side: GROUP BY vs window function
print('=== GROUP BY: one row per account ===')
q_group = """
SELECT  account_id,
        COUNT(*)               AS txn_count,
        ROUND(SUM(amount), 2)  AS total_amount
FROM    transactions
GROUP BY account_id
ORDER BY account_id
"""
print(pd.read_sql(q_group, conn).to_string(index=False))

print()
print('=== Window function: every row kept, group total appended ===')
q_window = """
SELECT  txn_id,
        account_id,
        txn_date,
        amount,
        COUNT(*)              OVER (PARTITION BY account_id) AS acct_txn_count,
        ROUND(SUM(amount)     OVER (PARTITION BY account_id), 2) AS acct_total,
        ROUND(amount * 100.0 /
              SUM(ABS(amount)) OVER (PARTITION BY account_id), 1) AS pct_of_acct_volume
FROM    transactions
ORDER BY account_id, txn_date
"""
print(pd.read_sql(q_window, conn).to_string(index=False))


=== GROUP BY: one row per account ===
 account_id  txn_count  total_amount
        101          6       10650.0
        103          5       39000.0
        105          4        4850.0
        107          3        3325.0
        109          2        2150.0
        111          3        7900.0

=== Window function: every row kept, group total appended ===
 txn_id  account_id   txn_date  amount  acct_txn_count  acct_total  pct_of_acct_volume
   1001         101 2023-01-05  4200.0               6     10650.0                28.9
   1002         101 2023-01-12  -850.0               6     10650.0                -5.8
   1003         101 2023-01-20  -120.0               6     10650.0                -0.8
   1004         101 2023-02-05  4200.0               6     10650.0                28.9
   1005         101 2023-02-18  -980.0               6     10650.0                -6.7
   1006         101 2023-03-05  4200.0               6     10650.0                28.9
   1007         103 2023-01-08 

---
## PARTITION BY — dividing the window into groups

In [3]:
# Without PARTITION BY: window covers all rows
print('=== No PARTITION BY — window is the entire result set ===')
q = """
SELECT  txn_id,
        account_id,
        txn_date,
        amount,
        ROUND(SUM(amount) OVER (), 2)          AS grand_total,
        ROUND(amount * 100.0 /
              SUM(ABS(amount)) OVER (), 1)     AS pct_of_all_txns
FROM    transactions
ORDER BY txn_date
LIMIT 8
"""
print(pd.read_sql(q, conn).to_string(index=False))

# With PARTITION BY account_id
print()
print('=== With PARTITION BY account_id — window resets per account ===')
q2 = """
SELECT  txn_id,
        account_id,
        amount,
        ROUND(SUM(amount) OVER (PARTITION BY account_id), 2)  AS account_total,
        ROUND(SUM(amount) OVER (),                      2)    AS grand_total,
        ROUND(amount * 100.0 /
              SUM(ABS(amount)) OVER (PARTITION BY account_id), 1) AS pct_of_account
FROM    transactions
ORDER BY account_id, txn_date
LIMIT 10
"""
print(pd.read_sql(q2, conn).to_string(index=False))


=== No PARTITION BY — window is the entire result set ===
 txn_id  account_id   txn_date  amount  grand_total  pct_of_all_txns
   1001         101 2023-01-05  4200.0      67875.0              4.6
   1012         105 2023-01-06  3100.0      67875.0              3.4
   1007         103 2023-01-08 15000.0      67875.0             16.5
   1019         109 2023-01-10  2800.0      67875.0              3.1
   1002         101 2023-01-12  -850.0      67875.0             -0.9
   1013         105 2023-01-19  -700.0      67875.0             -0.8
   1003         101 2023-01-20  -120.0      67875.0             -0.1
   1016         107 2023-01-20  3800.0      67875.0              4.2

=== With PARTITION BY account_id — window resets per account ===
 txn_id  account_id  amount  account_total  grand_total  pct_of_account
   1001         101  4200.0        10650.0      67875.0            28.9
   1002         101  -850.0        10650.0      67875.0            -5.8
   1003         101  -120.0        1065

---
## ORDER BY inside OVER — introducing sequence

In [4]:
# ORDER BY inside OVER changes the function from a full-partition aggregate
# to a running/cumulative aggregate
print('=== SUM OVER without ORDER BY: full partition total (same value per row) ===')
q1 = """
SELECT  txn_id, account_id, txn_date, amount,
        ROUND(SUM(amount) OVER (PARTITION BY account_id), 2) AS running_or_total
FROM    transactions
WHERE   account_id = 101
ORDER BY txn_date
"""
print(pd.read_sql(q1, conn).to_string(index=False))

print()
print('=== SUM OVER with ORDER BY: running (cumulative) total — value grows row by row ===')
q2 = """
SELECT  txn_id, account_id, txn_date, amount,
        ROUND(SUM(amount) OVER (
            PARTITION BY account_id
            ORDER BY txn_date
        ), 2) AS running_total
FROM    transactions
WHERE   account_id = 101
ORDER BY txn_date
"""
print(pd.read_sql(q2, conn).to_string(index=False))
print()
print('Key insight: adding ORDER BY to OVER turns a total into a running total.')
print('The default frame with ORDER BY is RANGE BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW.')


=== SUM OVER without ORDER BY: full partition total (same value per row) ===
 txn_id  account_id   txn_date  amount  running_or_total
   1001         101 2023-01-05  4200.0           10650.0
   1002         101 2023-01-12  -850.0           10650.0
   1003         101 2023-01-20  -120.0           10650.0
   1004         101 2023-02-05  4200.0           10650.0
   1005         101 2023-02-18  -980.0           10650.0
   1006         101 2023-03-05  4200.0           10650.0

=== SUM OVER with ORDER BY: running (cumulative) total — value grows row by row ===
 txn_id  account_id   txn_date  amount  running_total
   1001         101 2023-01-05  4200.0         4200.0
   1002         101 2023-01-12  -850.0         3350.0
   1003         101 2023-01-20  -120.0         3230.0
   1004         101 2023-02-05  4200.0         7430.0
   1005         101 2023-02-18  -980.0         6450.0
   1006         101 2023-03-05  4200.0        10650.0

Key insight: adding ORDER BY to OVER turns a total into a ru

---
## Named windows with WINDOW clause

In [5]:
# When multiple window functions share the same definition, use WINDOW to name it
# Avoids repeating the OVER(...) specification
print('=== Named window (WINDOW clause) — avoids repeating OVER definitions ===')
q = """
SELECT  txn_id,
        account_id,
        txn_date,
        amount,
        ROW_NUMBER() OVER w                        AS row_num,
        ROUND(SUM(amount)  OVER w, 2)              AS running_total,
        ROUND(AVG(amount)  OVER w, 2)              AS running_avg,
        COUNT(*)           OVER w                  AS running_count
FROM    transactions
WHERE   account_id IN (101, 103)
WINDOW  w AS (PARTITION BY account_id ORDER BY txn_date)
ORDER BY account_id, txn_date
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('WINDOW clause works in both SQLite (3.25+) and PostgreSQL.')
print('It makes queries with many window functions significantly more readable.')

conn.close()


=== Named window (WINDOW clause) — avoids repeating OVER definitions ===
 txn_id  account_id   txn_date  amount  row_num  running_total  running_avg  running_count
   1001         101 2023-01-05  4200.0        1         4200.0      4200.00              1
   1002         101 2023-01-12  -850.0        2         3350.0      1675.00              2
   1003         101 2023-01-20  -120.0        3         3230.0      1076.67              3
   1004         101 2023-02-05  4200.0        4         7430.0      1857.50              4
   1005         101 2023-02-18  -980.0        5         6450.0      1290.00              5
   1006         101 2023-03-05  4200.0        6        10650.0      1775.00              6
   1007         103 2023-01-08 15000.0        1        15000.0     15000.00              1
   1008         103 2023-01-25 -3200.0        2        11800.0      5900.00              2
   1009         103 2023-02-08 15000.0        3        26800.0      8933.33              3
   1010         1

---
## Common Pitfalls

**1. Confusing window functions with GROUP BY**
Window functions never remove rows. If you expect one row per account, you need GROUP BY. If you want per-row values alongside group-level context (e.g. each transaction's share of its account total), you need a window function. Mixing both in the same query is valid — aggregate first in a subquery or CTE, then apply window functions.

**2. Forgetting that ORDER BY inside OVER changes the frame**
`SUM(amount) OVER (PARTITION BY account_id)` returns the full partition total for every row. `SUM(amount) OVER (PARTITION BY account_id ORDER BY txn_date)` returns the running total up to the current row. Adding ORDER BY silently changes the behaviour — always be intentional.

**3. Window functions execute after WHERE, GROUP BY, and HAVING**
You cannot use a window function result in a WHERE clause directly. `WHERE ROW_NUMBER() OVER (...) = 1` raises an error. Wrap the window function in a subquery or CTE, then filter on the result in the outer query.

**4. Ambiguous ORDER BY when multiple rows share the same date**
`ROW_NUMBER() OVER (ORDER BY txn_date)` assigns arbitrary ranks to rows with identical dates. Add a tiebreaker to the ORDER BY: `ORDER BY txn_date, txn_id` for deterministic results.

**5. NULL handling in PARTITION BY and ORDER BY**
NULL values in PARTITION BY columns form their own partition (all NULLs grouped together). NULL values in ORDER BY sort last by default in SQLite and PostgreSQL (ascending). These are usually correct behaviours, but verify when your data has meaningful NULLs in partitioning or ordering columns.

**6. Performance — window functions scan the partition for each row**
A window function with `PARTITION BY account_id ORDER BY txn_date` sorts and scans data per partition. On large tables, ensure the partitioning and ordering columns are indexed. In PostgreSQL, `EXPLAIN ANALYZE` will show whether the window sort is using an index or a full sort step.


---
*sql_methods_library - Samantha McGarrigle*